# Task 2: Support Vector Machine (SVM) for Classification
Implement a Support Vector Machine (SVM) model for binary classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

### 1. Generating and Loading Dataset

In [ ]:
def make_dataset():
    # Creating a synthetic dataset designed to show differences between kernels
    X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
    return X, y

X, y = make_dataset()

# Converting to pandas dataframe
df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])
df['Target'] = y
display(df.head())

### 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

### 3. Model Training (Linear vs RBF)

In [ ]:
svm_linear = SVC(kernel='linear', probability=True, random_state=42)
svm_linear.fit(X_train, y_train)

svm_rbf = SVC(kernel='rbf', probability=True, random_state=42)
svm_rbf.fit(X_train, y_train)

### 4. Evaluate Models (Accuracy, Precision, Recall, AUC)

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(f"--- Evaluation Metrics for {model_name} ---")
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"AUC:       {auc:.4f}\n")
    return accuracy, precision, recall, auc

# Linear Predictions
y_pred_linear = svm_linear.predict(X_test)
y_prob_linear = svm_linear.predict_proba(X_test)[:, 1]
evaluate_model(y_test, y_pred_linear, y_prob_linear, "SVM (Linear Kernel)")

# RBF Predictions
y_pred_rbf = svm_rbf.predict(X_test)
y_prob_rbf = svm_rbf.predict_proba(X_test)[:, 1]
evaluate_model(y_test, y_pred_rbf, y_prob_rbf, "SVM (RBF Kernel)")

### 5. Visualizing Decision Boundaries and Metrics

In [ ]:
def plot_decision_boundary(clf, X, y, title):
    # Set min and max values and give it some padding
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    h = 0.02
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k')
    plt.title(title)

plt.figure(figsize=(15, 10))

# Decision Boundary for Linear
plt.subplot(2, 2, 1)
plot_decision_boundary(svm_linear, X_train, y_train, "Decision Boundary: Linear Kernel")

# Decision Boundary for RBF
plt.subplot(2, 2, 2)
plot_decision_boundary(svm_rbf, X_train, y_train, "Decision Boundary: RBF Kernel")

# ROC Curves
plt.subplot(2, 2, 3)
fpr_linear, tpr_linear, _ = roc_curve(y_test, y_prob_linear)
fpr_rbf, tpr_rbf, _ = roc_curve(y_test, y_prob_rbf)
plt.plot(fpr_linear, tpr_linear, label=f'Linear (AUC = {roc_auc_score(y_test, y_prob_linear):.2f})')
plt.plot(fpr_rbf, tpr_rbf, label=f'RBF (AUC = {roc_auc_score(y_test, y_prob_rbf):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()

# Confusion Matrix (RBF)
plt.subplot(2, 2, 4)
cm = confusion_matrix(y_test, y_pred_rbf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svm_rbf.classes_)
disp.plot(ax=plt.gca(), cmap='Blues')
plt.title("Confusion Matrix: RBF Kernel")

plt.tight_layout()
plt.show()